# Laboratorio: diagonalización y proyectores espectrales

Construiremos diagonalizaciones exactas, diagnosticaremos matrices no diagonalizables y aplicaremos la descomposición espectral a potencias, exponenciales y sistemas dinámicos.

## 0. Preparación

In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt

sp.init_printing()
np.set_printoptions(precision=6, suppress=True)

## 1. Una función para diagonalizar exactamente

Reuniremos bases de todos los espacios propios. La matriz es diagonalizable si y solo si el número total de vectores obtenidos es igual al orden de la matriz.

In [ ]:
def diagonalizacion_exacta(A):
    A = sp.Matrix(A)
    if A.rows != A.cols:
        raise ValueError("La matriz debe ser cuadrada.")
    columnas, diagonal, datos = [], [], []
    for lam, mult_alg, base in A.eigenvects():
        mult_geom = len(base)
        datos.append((lam, mult_alg, mult_geom, base))
        columnas.extend(base)
        diagonal.extend([lam] * mult_geom)
    if len(columnas) != A.rows:
        return None, None, datos
    P = sp.Matrix.hstack(*columnas)
    D = sp.diag(*diagonal)
    assert P.det() != 0
    assert A*P == P*D
    assert sp.simplify(P.inv()*A*P) == D
    return P, D, datos

## 2. Ejemplo de las presentaciones

Diagonalizamos $A=\begin{pmatrix}7&1\\2&8\end{pmatrix}$. Sus valores propios son distintos, por lo que la diagonalización existe.

In [ ]:
A = sp.Matrix([[7, 1], [2, 8]])
P, D, datos = diagonalizacion_exacta(A)
print("Datos (lambda, ma, mg, base):")
for dato in datos:
    print(dato)
print("P =")
sp.pprint(P)
print("D =")
sp.pprint(D)
assert set(D.diagonal()) == {6, 9}

## 3. Un valor repetido no impide diagonalizar

La multiplicidad algebraica de $1$ es dos. Como su espacio propio también tiene dimensión dos, se obtienen suficientes vectores propios.

In [ ]:
S = sp.Matrix([[2, 1, 1], [1, 2, 1], [1, 1, 2]])
P_S, D_S, datos_S = diagonalizacion_exacta(S)
for lam, ma, mg, base in datos_S:
    print(f"lambda={lam}: ma={ma}, mg={mg}, base={base}")
print("P^{-1} S P =")
sp.pprint(P_S.inv()*S*P_S)
assert P_S is not None
assert sorted(D_S.diagonal()) == [1, 1, 4]

## 4. Diagnóstico de una matriz no diagonalizable

Para el valor propio $4$ faltará una dirección propia: su multiplicidad geométrica será menor que su multiplicidad algebraica.

In [ ]:
B = sp.Matrix([[4, 1, 0], [0, 4, 0], [0, 0, 2]])
P_B, D_B, datos_B = diagonalizacion_exacta(B)
for lam, ma, mg, base in datos_B:
    print(f"lambda={lam}: ma={ma}, mg={mg}, base={base}")
assert P_B is None and D_B is None
assert any(lam == 4 and ma == 2 and mg == 1 for lam, ma, mg, _ in datos_B)

## 5. Potencias mediante diagonalización

Comparamos el cálculo directo de $A^{12}$ con $PD^{12}P^{-1}$. La segunda expresión convierte la potencia matricial en potencias escalares.

In [ ]:
k = 12
A12_directa = A**k
A12_diagonal = sp.simplify(P * D**k * P.inv())
assert A12_directa == A12_diagonal
print("A^12 =")
sp.pprint(A12_diagonal)

## 6. Proyectores espectrales

Para los valores propios $6$ y $9$, los proyectores se obtienen por interpolación. No es necesario normalizar vectores propios ni invertir una matriz de vectores.

In [ ]:
I2 = sp.eye(2)
Pi6 = sp.simplify((A - 9*I2)/(6 - 9))
Pi9 = sp.simplify((A - 6*I2)/(9 - 6))
print("Pi_6 =")
sp.pprint(Pi6)
print("Pi_9 =")
sp.pprint(Pi9)

assert Pi6**2 == Pi6 and Pi9**2 == Pi9
assert Pi6*Pi9 == sp.zeros(2) and Pi9*Pi6 == sp.zeros(2)
assert Pi6 + Pi9 == I2
assert 6*Pi6 + 9*Pi9 == A
assert 6**k*Pi6 + 9**k*Pi9 == A**k

## 7. Exponencial matricial

La descomposición espectral da $e^{tA}=e^{6t}\Pi_6+e^{9t}\Pi_9$. Verificamos que coincide con la fórmula obtenida usando $P$ y $D$.

In [ ]:
tau = sp.symbols('tau', real=True)
exp_por_P = sp.simplify(P * sp.diag(sp.exp(D[0,0]*tau), sp.exp(D[1,1]*tau)) * P.inv())
exp_por_proyectores = sp.simplify(sp.exp(6*tau)*Pi6 + sp.exp(9*tau)*Pi9)
assert sp.simplify(exp_por_P - exp_por_proyectores) == sp.zeros(2)
assert exp_por_proyectores.subs(tau, 0) == sp.eye(2)
assert sp.simplify(sp.diff(exp_por_proyectores, tau) - A*exp_por_proyectores) == sp.zeros(2)
sp.pprint(exp_por_proyectores)

## 8. Dinámica discreta y modos propios

Construimos una matriz con valores propios $4/5$ y $11/10$. El primer modo decae y el segundo crece. Las coordenadas propias permiten observar ambos comportamientos por separado.

In [ ]:
P_dyn = sp.Matrix([[1, 1], [0, 1]])
D_dyn = sp.diag(sp.Rational(4,5), sp.Rational(11,10))
A_dyn = sp.simplify(P_dyn * D_dyn * P_dyn.inv())
x0 = sp.Matrix([2, 1])
c0 = P_dyn.inv() * x0
print("A dinámica =")
sp.pprint(A_dyn)
print("Coordenadas propias iniciales:", list(c0))
for j in [0, 1, 5, 10]:
    x_directo = A_dyn**j * x0
    x_modal = P_dyn * D_dyn**j * c0
    assert sp.simplify(x_directo - x_modal) == sp.zeros(2, 1)

In [ ]:
pasos = np.arange(31)
coef_1 = float(c0[0]) * (4/5)**pasos
coef_2 = float(c0[1]) * (11/10)**pasos
trayectoria = np.array([
    np.array(A_dyn**int(j) * x0, dtype=float).reshape(-1) for j in pasos
])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].plot(pasos, np.abs(coef_1), label=r'$|c_1(4/5)^k|$', lw=2)
axes[0].plot(pasos, np.abs(coef_2), label=r'$|c_2(11/10)^k|$', lw=2)
axes[0].set(xlabel='paso k', ylabel='magnitud modal', yscale='log')
axes[0].grid(alpha=0.25)
axes[0].legend()

axes[1].plot(trayectoria[:,0], trayectoria[:,1], 'o-', ms=3, lw=1.5)
axes[1].scatter(*trayectoria[0], s=70, label='$x_0$')
axes[1].scatter(*trayectoria[-1], s=70, label='$x_{30}$')
axes[1].set(xlabel='$x_1$', ylabel='$x_2$', title='trayectoria en coordenadas originales')
axes[1].grid(alpha=0.25)
axes[1].legend()
plt.tight_layout()
plt.show()

## 9. Actividades

1. Cambia el orden de las columnas de $P$ y comprueba qué cambio debe hacerse en $D$.
2. Escala una columna de $P$ por un número no nulo y verifica que la matriz $A$ reconstruida no cambia.
3. Aplica la función de diagonalización a dos matrices con espectro $1,1,2$ y distintas multiplicidades geométricas.
4. Usa los proyectores espectrales para calcular $(A^2+I)^{-1}$.
5. Modifica los valores propios del sistema dinámico y clasifica el origen como estable, inestable o neutral.

## 10. Cierre

- Diagonalizar equivale a encontrar una base completa de vectores propios.
- Las multiplicidades geométricas indican si hay suficientes direcciones propias.
- Las funciones de una matriz diagonalizable se calculan sobre sus valores propios.
- Los proyectores espectrales separan los modos propios sin elegir una base particular dentro de cada espacio propio.
- La diagonalización general no implica ortogonalidad.